In [1]:
import pandas as pd

In [2]:
# Inspect train.csv
train_df = pd.read_csv('train.csv', nrows=5)
print("--- TRAIN.CSV HEAD ---")
print(train_df.head())
print("\n--- TRAIN.CSV INFO ---")
train_info = pd.read_csv('train.csv', nrows=1000).info()

# Inspect sample_submission.csv
sub_df = pd.read_csv('sample_submission.csv', nrows=5)
print("\n--- SAMPLE_SUBMISSION.CSV HEAD ---")
print(sub_df.head())

--- TRAIN.CSV HEAD ---
         Date      Stt   ItemCode  Quantity    UnitPrice  SalesAmount  \
0  2020-11-17  2000004  SKU-08063        12       242700      2184300   
1  2020-11-17  2000003  SKU-09458       600  131818,1818     79090909   
2  2020-11-18  2000007  SKU-08062         6       230000       940909   
3  2020-11-18  2000006  SKU-09458       240       270000     44181818   
4  2020-11-18  2000005  SKU-09458       240       270000     44181818   

  Unit Cost  Cost Amount  
0  123559,1      1482709  
1    110000     66000000  
2    101000       606000  
3    110000     26400000  
4    110000     26400000  

--- TRAIN.CSV INFO ---
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 8 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   Date         1000 non-null   object
 1   Stt          1000 non-null   int64 
 2   ItemCode     1000 non-null   object
 3   Quantity     1000 non-null   int64 
 

In [3]:
import pandas as pd
sub_df = pd.read_csv('sample_submission.csv')
print("Unique suffixes in id:", sub_df['id'].str.split('_').str[-1].unique())
print("Total rows:", len(sub_df))

Unique suffixes in id: ['validation' 'evaluation']
Total rows: 31944


# Model Score: 0.517
submission (3).csv

In [4]:
import pandas as pd
import numpy as np
import lightgbm as lgb
from datetime import timedelta
import warnings
warnings.filterwarnings('ignore')

# ---------------------------------------------------------
# CONFIGURATE STRICT TIMELINES (AVOID DATA LEAKAGES)
# ---------------------------------------------------------
TRAIN_START = '2023-01-01'
TRAIN_END   = '2025-08-08'
VALID_START = '2025-08-09'
VALID_END   = '2025-09-05'

PUBLIC_LB_START = '2025-09-06'
PUBLIC_LB_END   = '2025-10-03' # F1..F28
PRIVATE_LB_START= '2025-10-04'
PRIVATE_LB_END  = '2025-10-31' # F29..F56

def safe_data_loader(file_path):
    print("1. Loading and cleaning data")
    cols = ['Date', 'ItemCode', 'Quantity', 'SalesAmount', 'Cost Amount']
    df = pd.read_csv(file_path, usecols=cols)
    df['Date'] = pd.to_datetime(df['Date'])

    for col in ['SalesAmount', 'Cost Amount']:
        df[col] = df[col].astype(str).str.replace(',', '.').astype(float)

    df['Quantity'] = df['Quantity'].clip(lower=0)

    daily_sales = df.groupby(['Date', 'ItemCode'])['Quantity'].sum().reset_index()
    return df, daily_sales

def calculate_strict_wrmsse_weights(raw_df, daily_sales):
    print("2. Calculating WRMSSE (Strict Mode)")

    past_df = raw_df[raw_df['Date'] <= TRAIN_END]
    past_sales = daily_sales[daily_sales['Date'] <= TRAIN_END]

    #Calculate revenue on past dataset
    sku_fin = past_df.groupby('ItemCode')[['SalesAmount', 'Cost Amount']].sum()
    sku_fin['Profit'] = (sku_fin['SalesAmount'] - sku_fin['Cost Amount']).clip(lower=0)

    #Calculate Scale (RMSSE volatility) on past dataset
    pivot_sales = past_sales.pivot(index='Date', columns='ItemCode', values='Quantity').fillna(0)
    scale_dict = ((pivot_sales - pivot_sales.shift(1)) ** 2).mean().to_dict()

    weights = {}
    for item in raw_df['ItemCode'].unique():
        profit = sku_fin.loc[item, 'Profit'] if item in sku_fin.index else 0
        scale = scale_dict.get(item, 1)
        if scale == 0: scale = 0.1
        weights[item] = profit / (np.sqrt(scale) + 1e-4)

    mean_weight = np.mean(list(weights.values()))
    weight_dict = {k: v / (mean_weight + 1e-5) for k, v in weights.items()}

    #Classification of Dead SKUs (Not sold in 150 days, part of the Train series)
    recent_past_sales = past_sales[past_sales['Date'] >= pd.to_datetime(TRAIN_END) - timedelta(days=150)]
    active_skus = recent_past_sales[recent_past_sales['Quantity'] > 0]['ItemCode'].unique()
    dead_skus = set(raw_df['ItemCode'].unique()) - set(active_skus)

    return weight_dict, dead_skus

def build_grid_and_features(daily_sales, weight_dict, base_lag):
    print(f"3. Building Grid & Features (Base Lag = {base_lag})")

    all_skus = daily_sales['ItemCode'].unique()

    date_range = pd.date_range(TRAIN_START, PRIVATE_LB_END)
    grid = pd.MultiIndex.from_product([date_range, all_skus], names=['Date', 'ItemCode']).to_frame(index=False)

    df = pd.merge(grid, daily_sales, on=['Date', 'ItemCode'], how='left')
    df['Quantity'] = df['Quantity'].fillna(0).astype(np.float32)
    df.sort_values(['ItemCode', 'Date'], inplace=True)
    df.reset_index(drop=True, inplace=True)

    df['ItemCode_cat'] = df['ItemCode'].astype('category')
    df['Weight'] = df['ItemCode'].map(weight_dict).fillna(0.01).astype(np.float32)

    df['dayofweek'] = df['Date'].dt.dayofweek.astype('int8')
    df['month'] = df['Date'].dt.month.astype('int8')

    #AVOID DATA LEAKAGES
    df[f'lag_{base_lag}'] = df.groupby('ItemCode')['Quantity'].shift(base_lag).astype(np.float32)
    df[f'lag_{base_lag+7}'] = df.groupby('ItemCode')['Quantity'].shift(base_lag + 7).astype(np.float32)

    df['roll_mean_7'] = df.groupby('ItemCode')[f'lag_{base_lag}'].transform(lambda x: x.rolling(7).mean()).astype(np.float32)
    df['roll_mean_28'] = df.groupby('ItemCode')[f'lag_{base_lag}'].transform(lambda x: x.rolling(28).mean()).astype(np.float32)
    df['zero_prop_28'] = df.groupby('ItemCode')[f'lag_{base_lag}'].transform(lambda x: (x == 0).rolling(28).mean()).astype(np.float32)

    return df

def train_and_predict():
    raw_df, daily_sales = safe_data_loader('train.csv')
    weight_dict, dead_skus = calculate_strict_wrmsse_weights(raw_df, daily_sales)

    lgb_params = {
        'objective': 'tweedie',
        'tweedie_variance_power': 1.1,
        'metric': 'rmse',
        'learning_rate': 0.05,
        'num_leaves': 63,
        'min_data_in_leaf': 150,
        'feature_fraction': 0.8,
        'n_estimators': 1500,
        'verbose': -1,
        'seed': 42
    }

    # ==============================================================
    # MODEL 1: FORECAST PUBLIC LB (Lag 28)
    # ==============================================================
    print("\n--- STARTING MODEL 1 (PUBLIC LB) ---")
    df_m1 = build_grid_and_features(daily_sales, weight_dict, base_lag=28)
    features_m1 = ['ItemCode_cat', 'dayofweek', 'month', 'lag_28', 'lag_35', 'roll_mean_7', 'roll_mean_28', 'zero_prop_28']

    train_mask_1 = (df_m1['Date'] >= '2023-08-01') & (df_m1['Date'] <= TRAIN_END)
    valid_mask_1 = (df_m1['Date'] >= VALID_START) & (df_m1['Date'] <= VALID_END)

    train_data_1 = lgb.Dataset(df_m1[train_mask_1][features_m1], label=df_m1[train_mask_1]['Quantity'], weight=df_m1[train_mask_1]['Weight'], categorical_feature=['ItemCode_cat'])
    valid_data_1 = lgb.Dataset(df_m1[valid_mask_1][features_m1], label=df_m1[valid_mask_1]['Quantity'], weight=df_m1[valid_mask_1]['Weight'], categorical_feature=['ItemCode_cat'])

    model_1 = lgb.train(lgb_params, train_data_1, valid_sets=[train_data_1, valid_data_1], callbacks=[lgb.early_stopping(50), lgb.log_evaluation(200)])

    pub_pred_mask = (df_m1['Date'] >= PUBLIC_LB_START) & (df_m1['Date'] <= PUBLIC_LB_END)
    df_m1.loc[pub_pred_mask, 'Quantity'] = model_1.predict(df_m1[pub_pred_mask][features_m1])

    # ==============================================================
    # MODEL 2: FORECAST PRIVATE LB (Lag 56)
    # ==============================================================
    print("\n--- STARTING MODEL 2 (PRIVATE LB) ---")

    df_m2 = build_grid_and_features(daily_sales, weight_dict, base_lag=56)
    features_m2 = ['ItemCode_cat', 'dayofweek', 'month', 'lag_56', 'lag_63', 'roll_mean_7', 'roll_mean_28', 'zero_prop_28']

    #Cut back 28 days compared to model 1.
    train_mask_2 = (df_m2['Date'] >= '2023-09-01') & (df_m2['Date'] <= '2025-07-11')
    valid_mask_2 = (df_m2['Date'] >= '2025-07-12') & (df_m2['Date'] <= VALID_END)

    train_data_2 = lgb.Dataset(df_m2[train_mask_2][features_m2], label=df_m2[train_mask_2]['Quantity'], weight=df_m2[train_mask_2]['Weight'], categorical_feature=['ItemCode_cat'])
    valid_data_2 = lgb.Dataset(df_m2[valid_mask_2][features_m2], label=df_m2[valid_mask_2]['Quantity'], weight=df_m2[valid_mask_2]['Weight'], categorical_feature=['ItemCode_cat'])

    model_2 = lgb.train(lgb_params, train_data_2, valid_sets=[train_data_2, valid_data_2], callbacks=[lgb.early_stopping(50), lgb.log_evaluation(200)])

    priv_pred_mask = (df_m2['Date'] >= PRIVATE_LB_START) & (df_m2['Date'] <= PRIVATE_LB_END)
    df_m2.loc[priv_pred_mask, 'Quantity'] = model_2.predict(df_m2[priv_pred_mask][features_m2])

    # ==============================================================
    # PROCESSING AND EXPORTING FILE
    # ==============================================================
    print("\n4. Apply Thresholding and Submission Consolidation")
    #Prediction of model 1 for F1-F28
    val_df = df_m1[pub_pred_mask].copy()
    val_df.loc[val_df['Quantity'] < 0.25, 'Quantity'] = 0
    val_df.loc[val_df['ItemCode'].isin(dead_skus), 'Quantity'] = 0

    val_df['F_day'] = 'F' + (val_df['Date'] - pd.to_datetime(PUBLIC_LB_START) + timedelta(days=1)).dt.days.astype(str)
    val_pivot = val_df.pivot(index='ItemCode', columns='F_day', values='Quantity').reset_index()
    val_pivot['id'] = val_pivot['ItemCode'] + '_validation'

    #Prediction of model 2 for F29-F56
    eval_df = df_m2[priv_pred_mask].copy()
    eval_df.loc[eval_df['Quantity'] < 0.25, 'Quantity'] = 0
    eval_df.loc[eval_df['ItemCode'].isin(dead_skus), 'Quantity'] = 0

    eval_df['F_day'] = 'F' + (eval_df['Date'] - pd.to_datetime(PRIVATE_LB_START) + timedelta(days=1)).dt.days.astype(str)
    eval_pivot = eval_df.pivot(index='ItemCode', columns='F_day', values='Quantity').reset_index()
    eval_pivot['id'] = eval_pivot['ItemCode'] + '_evaluation'

    #Merge
    cols_order = ['id'] + [f'F{i}' for i in range(1, 29)]
    final_sub = pd.concat([val_pivot, eval_pivot], ignore_index=True)[cols_order]

    sample_sub = pd.read_csv('sample_submission.csv')
    final_sub = pd.merge(sample_sub[['id']], final_sub, on='id', how='left').fillna(0)

    final_sub.to_csv('submission (3).csv', index=False)
    print("submission (3).csv is successfully exported")

if __name__ == "__main__":
    train_and_predict()

1. Loading and cleaning data
2. Calculating WRMSSE (Strict Mode)

--- STARTING MODEL 1 (PUBLIC LB) ---
3. Building Grid & Features (Base Lag = 28)
Training until validation scores don't improve for 50 rounds
Early stopping, best iteration is:
[66]	training's rmse: 3.00973	valid_1's rmse: 2.00932

--- STARTING MODEL 2 (PRIVATE LB) ---
3. Building Grid & Features (Base Lag = 56)
Training until validation scores don't improve for 50 rounds
Early stopping, best iteration is:
[75]	training's rmse: 2.91397	valid_1's rmse: 1.94193

4. Apply Thresholding and Submission Consolidation
submission (3).csv is successfully exported
